# DKT

Student-blocked nested evaluation of DKT on two binary prediction tasks:

- **First attempt:** `firstattemptcorrect`
- **After feedback:** `eventualcorrect`

Run the setup cell, then either task cell. Each task uses its own fixed outer-fold file and output directory. Hyperparameters and the final training epoch count are selected exclusively from inner validation folds.


In [ ]:
from pathlib import Path
import pandas as pd

# Point this to the interaction-level FeedBook export.
DATA_PATH = Path("data/feedbook_interactions.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}. Update DATA_PATH before running."
    )

logging_data = pd.read_csv(DATA_PATH)
print(f"Loaded {len(logging_data):,} interactions from {DATA_PATH}")


## First-attempt prediction


In [ ]:
# DKT First Attempt

# This wrapper follows the pyKT DKT alignment:
#   y_t = DKT(q_<=t, r_<=t) gives a probability for every KC,
#   select y_t[next_q] and compare it with next_r.
#
# Final training duration is selected from the inner validation folds.
# The held-out outer fold is used only for final OOF evaluation.

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    root_mean_squared_error,
    mean_absolute_error,
)
import optuna
from optuna.samplers import GridSampler
from statistics import mean, stdev

from pykt.models import dkt
DKT = dkt.DKT


# ============================================================
# 1. Reproducibility & device
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# 2. Configuration
# ============================================================
seq_len = 20
batch_size = 8
n_splits = 3
max_epochs = 100
patience = 10

model_name = "DKT"
kc_name = "propertyexercisetype"          # change per KC run
question_col = "KC (propertyexercisetype)" # change per KC run
correct_col = "firstattemptcorrect"
student_col = "Anon Student Id"

fixed_fold_file = "data/fixed_outer_folds.csv"
oof_output_dir = "DKT_FA"
os.makedirs(oof_output_dir, exist_ok=True)

# logging_data must already be loaded before running this script/cell.
# Example:
# logging_data = pd.read_csv("feedbook_firstattempt.csv")


# ============================================================
# 3. Preserve original row IDs BEFORE model-specific filtering
# ============================================================
logging_data = logging_data.copy().reset_index(drop=True)

if "row_id" not in logging_data.columns:
    logging_data["row_id"] = np.arange(len(logging_data), dtype=np.int64)

# Avoid merge suffixes if the same in-memory dataframe was used previously.
if "outer_fold" in logging_data.columns:
    logging_data = logging_data.drop(columns=["outer_fold"])


# ============================================================
# 4. Load/create fixed STUDENT-GROUPED outer folds
# ============================================================
if os.path.exists(fixed_fold_file):
    fixed_folds = pd.read_csv(fixed_fold_file)

    required = {"row_id", "outer_fold"}
    if not required.issubset(fixed_folds.columns):
        raise ValueError(
            f"{fixed_fold_file} must contain columns 'row_id' and 'outer_fold'."
        )

    if fixed_folds["row_id"].duplicated().any():
        raise ValueError(f"{fixed_fold_file} contains duplicate row_id values.")

    found_folds = sorted(
        fixed_folds["outer_fold"].dropna().astype(int).unique().tolist()
    )
    expected_folds = list(range(1, n_splits + 1))
    if found_folds != expected_folds:
        raise ValueError(
            f"{fixed_fold_file} contains folds {found_folds}, "
            f"but this run expects {expected_folds}."
        )

    logging_data = logging_data.merge(
        fixed_folds[["row_id", "outer_fold"]],
        on="row_id",
        how="left",
        validate="one_to_one",
    )

    if logging_data["outer_fold"].isna().any():
        raise ValueError(
            "Some row_id values are missing outer_fold assignments. "
            "The fixed-fold file may belong to a different dataset."
        )

    logging_data["outer_fold"] = logging_data["outer_fold"].astype(int)
    print(f"Loaded fixed folds from {fixed_fold_file}")

else:
    base_for_folds = logging_data.dropna(subset=[student_col]).copy()

    gkf = GroupKFold(n_splits=n_splits)
    base_for_folds["outer_fold"] = -1

    for fold_idx, (_, test_idx) in enumerate(
        gkf.split(base_for_folds, groups=base_for_folds[student_col]),
        start=1,
    ):
        base_for_folds.iloc[
            test_idx,
            base_for_folds.columns.get_loc("outer_fold"),
        ] = fold_idx

    fixed_folds = base_for_folds[["row_id", "outer_fold"]].copy()
    fixed_folds.to_csv(fixed_fold_file, index=False)

    logging_data = logging_data.merge(
        fixed_folds,
        on="row_id",
        how="left",
        validate="one_to_one",
    )

    print(f"Saved fixed folds to {fixed_fold_file}")


# ============================================================
# 5. Model-specific preprocessing
# ============================================================
logging_model = logging_data.dropna(
    subset=[question_col, correct_col, student_col, "outer_fold"]
).copy()

logging_model[correct_col] = logging_model[correct_col].astype(int)
logging_model["outer_fold"] = logging_model["outer_fold"].astype(int)

if not logging_model[correct_col].isin([0, 1]).all():
    raise ValueError(f"{correct_col} must contain only 0/1 values after filtering.")

# Validate that a student never appears in more than one fixed outer fold.
student_fold_counts = logging_model.groupby(student_col)["outer_fold"].nunique()
if (student_fold_counts > 1).any():
    bad_students = student_fold_counts[student_fold_counts > 1].index.tolist()[:10]
    raise ValueError(
        "Fixed outer folds are not student-grouped. Example offending students: "
        f"{bad_students}"
    )

# row_id -> student ID for OOF files
student_id_lookup = (
    logging_model[["row_id", student_col]]
    .drop_duplicates(subset=["row_id"])
    .set_index("row_id")[student_col]
    .to_dict()
)

# Global KC/question mapping. Reserve 0 for padding.
all_qids = logging_model[question_col].dropna().unique()
qid_to_index = {qid: idx + 1 for idx, qid in enumerate(all_qids)}
num_questions = len(qid_to_index) + 1

print(f"Model: {model_name}")
print(f"KC: {kc_name}")
print(f"Target: {correct_col}")
print(f"seq_len: {seq_len}")
print(f"num_questions (including padding 0): {num_questions}")


# ============================================================
# 6. Dataset
# ============================================================
class KTDataFromLogging(Dataset):
    def __init__(
        self,
        df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index,
    ):
        self.seq_len = seq_len
        self.samples = []

        df = df.copy()
        df["qid_index"] = (
            df[question_col].map(qid_to_index).fillna(0).astype(int)
        )

        # Preserve existing row order within each student.
        # Make sure logging_data is already in chronological interaction order.
        for _, group in df.groupby(student_col, sort=False):
            q_seq = group["qid_index"].tolist()
            r_seq = group[correct_col].astype(int).tolist()
            row_seq = group["row_id"].astype(int).tolist()

            # Adjacent windows share one interaction so every target retains context.
            for start in range(0, len(q_seq), seq_len - 1):
                end = min(start + seq_len, len(q_seq))

                if end - start < 2:
                    break

                q_chunk = q_seq[start:end]
                r_chunk = r_seq[start:end]
                row_chunk = row_seq[start:end]

                pad_len = seq_len - len(q_chunk)
                if pad_len > 0:
                    q_chunk += [0] * pad_len
                    r_chunk += [0] * pad_len
                    row_chunk += [-1] * pad_len

                self.samples.append(
                    (
                        torch.tensor(q_chunk, dtype=torch.long),
                        torch.tensor(r_chunk, dtype=torch.long),
                        torch.tensor(row_chunk, dtype=torch.long),
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# ============================================================
# 7. DKT helpers: pyKT-equivalent next-interaction alignment
# ============================================================
def dkt_selected_predictions(model, q_batch, r_batch):
    """
    outputs[:, t, k] = probability after interaction t for KC k.
    For target interaction t+1, select KC q[t+1] from output t.
    """
    outputs = model(q_batch, r_batch)  # [B, L, num_c]
    next_q = q_batch[:, 1:]            # [B, L-1]
    selected = outputs[:, :-1, :].gather(
        dim=2,
        index=next_q.unsqueeze(-1),
    ).squeeze(-1)                       # [B, L-1]
    return selected, next_q


def collect_predictions(model, data_loader, include_rows=False):
    model.eval()
    preds, labels, row_ids = [], [], []

    with torch.no_grad():
        for q_batch, r_batch, rowid_batch in data_loader:
            q_batch = q_batch.to(device)
            r_batch = r_batch.to(device)

            selected, next_q = dkt_selected_predictions(
                model, q_batch, r_batch
            )
            next_r = r_batch[:, 1:]
            valid_mask = next_q != 0

            if valid_mask.any():
                preds.extend(selected[valid_mask].detach().cpu().numpy().tolist())
                labels.extend(next_r[valid_mask].detach().cpu().numpy().tolist())

                if include_rows:
                    row_mask_cpu = valid_mask.detach().cpu()
                    next_rows = rowid_batch[:, 1:]
                    these_rows = next_rows[row_mask_cpu].numpy().tolist()
                    if any(rid == -1 for rid in these_rows):
                        raise RuntimeError("A padded row_id passed the valid DKT mask.")
                    row_ids.extend(these_rows)

    if include_rows:
        return preds, labels, row_ids
    return preds, labels


def safe_auc(labels, preds):
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return np.nan
    return roc_auc_score(labels, preds)


def compute_metrics(labels, preds):
    labels = np.asarray(labels, dtype=int)
    preds = np.asarray(preds, dtype=float)
    binary = (preds > 0.5).astype(int)

    return {
        "auc": safe_auc(labels, preds),
        "accuracy": accuracy_score(labels, binary),
        "rmse": root_mean_squared_error(labels, preds),
        "mae": mean_absolute_error(labels, preds),
        "precision": precision_score(labels, binary, zero_division=0),
        "recall": recall_score(labels, binary, zero_division=0),
        "f1": f1_score(labels, binary, zero_division=0),
    }


# ============================================================
# 8. Hyperparameter grid (unchanged from supplied DKT code)
# ============================================================
param_search_space = {
    "emb_size": [64, 128, 256],
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "dropout": [0.1, 0.3, 0.5],
}


# ============================================================
# 9. OOF and metric holders
# ============================================================
all_preds_all_folds = []
all_labels_all_folds = []
all_rowids_all_folds = []
all_foldnums_all_folds = []

auc_per_fold = []
acc_per_fold = []
rmse_per_fold = []
mae_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []


# ============================================================
# 10. Outer CV using fixed folds
# ============================================================
for fold in range(1, n_splits + 1):
    print(f"\n=== Outer Fold {fold}/{n_splits} ===")

    train_val_df = logging_model[
        logging_model["outer_fold"] != fold
    ].copy()

    test_df = logging_model[
        logging_model["outer_fold"] == fold
    ].copy()

    # -------------------------
    # Inner CV hyperparameter tuning
    # -------------------------
    inner_cv = GroupKFold(n_splits=n_splits)
    inner_groups = train_val_df[student_col]

    def objective(trial):
        emb_size = trial.suggest_categorical(
            "emb_size", param_search_space["emb_size"]
        )
        lr = trial.suggest_categorical(
            "learning_rate", param_search_space["learning_rate"]
        )
        dropout = trial.suggest_categorical(
            "dropout", param_search_space["dropout"]
        )

        auc_scores = []
        best_epochs = []

        for inner_train_idx, inner_val_idx in inner_cv.split(
            train_val_df,
            groups=inner_groups,
        ):
            inner_train_df = train_val_df.iloc[inner_train_idx]
            inner_val_df = train_val_df.iloc[inner_val_idx]

            train_dataset = KTDataFromLogging(
                inner_train_df,
                seq_len,
                question_col,
                correct_col,
                student_col,
                qid_to_index,
            )
            val_dataset = KTDataFromLogging(
                inner_val_df,
                seq_len,
                question_col,
                correct_col,
                student_col,
                qid_to_index,
            )

            train_loader = DataLoader(
                train_dataset,
                batch_size=batch_size,
                shuffle=True,
            )
            val_loader = DataLoader(
                val_dataset,
                batch_size=batch_size,
                shuffle=False,
            )

            model = DKT(
                num_c=num_questions,
                emb_size=emb_size,
                dropout=dropout,
                emb_type="qid",
            ).to(device)

            criterion = nn.BCELoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)

            best_auc_inner = -np.inf
            best_epoch_inner = 1
            no_improve = 0

            for epoch in range(max_epochs):
                model.train()

                for q_batch, r_batch, rowid_batch in train_loader:
                    q_batch = q_batch.to(device)
                    r_batch = r_batch.to(device)

                    optimizer.zero_grad()

                    selected_preds, next_q = dkt_selected_predictions(
                        model, q_batch, r_batch
                    )
                    next_r = r_batch[:, 1:].float()
                    valid_mask = next_q != 0

                    # A chunk of length 1 has no next-interaction target.
                    # Skip an all-singleton batch rather than taking BCE on empty tensors.
                    if not valid_mask.any():
                        continue

                    loss = criterion(
                        selected_preds[valid_mask],
                        next_r[valid_mask],
                    )

                    loss.backward()
                    optimizer.step()

                val_preds, val_labels = collect_predictions(
                    model, val_loader, include_rows=False
                )
                val_auc = safe_auc(val_labels, val_preds)

                if not np.isnan(val_auc):
                    if val_auc > best_auc_inner:
                        best_auc_inner = val_auc
                        best_epoch_inner = epoch + 1
                        no_improve = 0
                    else:
                        no_improve += 1

                    if no_improve >= patience:
                        break

            auc_scores.append(best_auc_inner)
            best_epochs.append(best_epoch_inner)

        trial.set_user_attr(
            "recommended_epochs",
            max(1, int(np.median(best_epochs))),
        )

        return np.mean(auc_scores) if auc_scores else 0.0

    sampler = GridSampler(param_search_space)
    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
    )

    grid_size = 1
    for values in param_search_space.values():
        grid_size *= len(values)

    study.optimize(
        objective,
        n_trials=grid_size,
        show_progress_bar=True,
    )

    best_params = study.best_params
    recommended_epochs = int(
        study.best_trial.user_attrs["recommended_epochs"]
    )
    print(
        f"\nBest params for fold {fold}: {best_params}, "
        f"AUC: {study.best_value:.6f}"
    )

    # -------------------------
    # Retrain best model on outer train+validation students
    # -------------------------
    train_dataset = KTDataFromLogging(
        train_val_df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index,
    )
    test_dataset = KTDataFromLogging(
        test_df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    best_model = DKT(
        num_c=num_questions,
        emb_size=best_params["emb_size"],
        dropout=best_params["dropout"],
        emb_type="qid",
    ).to(device)

    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(
        best_model.parameters(),
        lr=best_params["learning_rate"],
    )

    # Retrain on every outer-training student for the epoch count
    # selected from the inner validation folds.
    for epoch in range(recommended_epochs):
        best_model.train()

        for q_batch, r_batch, rowid_batch in train_loader:
            q_batch = q_batch.to(device)
            r_batch = r_batch.to(device)

            optimizer.zero_grad()

            selected_preds, next_q = dkt_selected_predictions(
                best_model, q_batch, r_batch
            )
            next_r = r_batch[:, 1:].float()
            valid_mask = next_q != 0

            if not valid_mask.any():
                continue

            loss = criterion(
                selected_preds[valid_mask],
                next_r[valid_mask],
            )

            loss.backward()
            optimizer.step()

        # The epoch count is selected only from inner validation folds.
        # The outer test fold is not inspected during training.

    # -------------------------
    # Final outer-test OOF predictions
    # -------------------------
    fold_preds, fold_labels, fold_row_ids = collect_predictions(
        best_model,
        test_loader,
        include_rows=True,
    )

    if len(fold_labels) == 0:
        print(f"WARNING: Fold {fold} produced no scorable predictions.")
        continue

    if len(fold_row_ids) != len(set(fold_row_ids)):
        raise ValueError(f"Duplicate row_id values inside OOF fold {fold}.")

    metrics = compute_metrics(fold_labels, fold_preds)

    auc_per_fold.append(metrics["auc"])
    acc_per_fold.append(metrics["accuracy"])
    rmse_per_fold.append(metrics["rmse"])
    mae_per_fold.append(metrics["mae"])
    precision_per_fold.append(metrics["precision"])
    recall_per_fold.append(metrics["recall"])
    f1_per_fold.append(metrics["f1"])

    print(f"\nEvaluation on Fold {fold} Test Set:")
    print(f"AUC: {metrics['auc']:.6f}")
    print(f"Accuracy: {metrics['accuracy']:.6f}")
    print(f"RMSE: {metrics['rmse']:.6f}")
    print(f"MAE: {metrics['mae']:.6f}")
    print(f"Precision: {metrics['precision']:.6f}")
    print(f"Recall: {metrics['recall']:.6f}")
    print(f"F1 Score: {metrics['f1']:.6f}")

    fold_df = pd.DataFrame({
        "row_id": fold_row_ids,
        "student_id": [student_id_lookup[row_id] for row_id in fold_row_ids],
        "fold": fold,
        "y_true": fold_labels,
        "y_pred": fold_preds,
        "model_name": model_name,
        "kc_name": kc_name,
    }).sort_values("row_id").reset_index(drop=True)

    fold_outfile = os.path.join(
        oof_output_dir,
        f"oof_{model_name}_{kc_name}_fold{fold}.csv",
    )
    fold_df.to_csv(fold_outfile, index=False)
    print(f"Saved Fold {fold} OOF predictions to {fold_outfile}")

    all_preds_all_folds.extend(fold_preds)
    all_labels_all_folds.extend(fold_labels)
    all_rowids_all_folds.extend(fold_row_ids)
    all_foldnums_all_folds.extend([fold] * len(fold_row_ids))


# ============================================================
# 11. Combined OOF
# ============================================================
oof_df = pd.DataFrame({
    "row_id": all_rowids_all_folds,
    "student_id": [student_id_lookup[row_id] for row_id in all_rowids_all_folds],
    "fold": all_foldnums_all_folds,
    "y_true": all_labels_all_folds,
    "y_pred": all_preds_all_folds,
    "model_name": model_name,
    "kc_name": kc_name,
}).sort_values("row_id").reset_index(drop=True)

if oof_df["row_id"].duplicated().any():
    dupes = oof_df.loc[oof_df["row_id"].duplicated(), "row_id"].head(10).tolist()
    raise ValueError(f"Duplicate row_id values in combined OOF: {dupes}")

# Check that saved fold number agrees with the fixed outer-fold assignment.
expected_fold_map = logging_model.set_index("row_id")["outer_fold"].to_dict()
wrong_fold = [
    row_id
    for row_id, fold in zip(oof_df["row_id"], oof_df["fold"])
    if expected_fold_map[row_id] != fold
]
if wrong_fold:
    raise ValueError(
        "OOF fold labels disagree with fixed outer folds. "
        f"Example row_ids: {wrong_fold[:10]}"
    )

oof_outfile = os.path.join(
    oof_output_dir,
    f"oof_{model_name}_{kc_name}_all.csv",
)
oof_df.to_csv(oof_outfile, index=False)
print(f"\nSaved combined OOF predictions to {oof_outfile}")


# ============================================================
# 12. Pooled + fold-average metrics
# ============================================================
if len(oof_df) > 0:
    pooled = compute_metrics(oof_df["y_true"].values, oof_df["y_pred"].values)

    print("\n=== Pooled OOF Results ===")
    print(f"Pooled AUC: {pooled['auc']:.6f}")
    print(f"Pooled Accuracy: {pooled['accuracy']:.6f}")
    print(f"Pooled RMSE: {pooled['rmse']:.6f}")
    print(f"Pooled MAE: {pooled['mae']:.6f}")
    print(f"Pooled Precision: {pooled['precision']:.6f}")
    print(f"Pooled Recall: {pooled['recall']:.6f}")
    print(f"Pooled F1: {pooled['f1']:.6f}")


def print_mean_sd(name, values):
    vals = [v for v in values if not np.isnan(v)]
    if not vals:
        print(f"{name}: NA")
    elif len(vals) == 1:
        print(f"{name}: {vals[0]:.6f}")
    else:
        print(f"{name}: {mean(vals):.6f} ± {stdev(vals):.6f}")


print(f"\n=== Fold-Averaged Results ({n_splits}-Fold CV) ===")
print_mean_sd("AUC", auc_per_fold)
print_mean_sd("Accuracy", acc_per_fold)
print_mean_sd("RMSE", rmse_per_fold)
print_mean_sd("MAE", mae_per_fold)
print_mean_sd("Precision", precision_per_fold)
print_mean_sd("Recall", recall_per_fold)
print_mean_sd("F1", f1_per_fold)

## After-feedback prediction


In [ ]:
# DKT After Feedback

# This wrapper follows the pyKT DKT alignment:
#   y_t = DKT(q_<=t, r_<=t) gives a probability for every KC,
#   select y_t[next_q] and compare it with next_r.
#
# Final training duration is selected from the inner validation folds.
# The held-out outer fold is used only for final OOF evaluation.

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    root_mean_squared_error,
    mean_absolute_error,
)
import optuna
from optuna.samplers import GridSampler
from statistics import mean, stdev

from pykt.models import dkt
DKT = dkt.DKT


# ============================================================
# 1. Reproducibility & device
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# 2. Configuration
# ============================================================
seq_len = 20
batch_size = 8
n_splits = 3
max_epochs = 100
patience = 10

model_name = "DKT"
kc_name = "actionableelementid"          # change per KC run
question_col = "KC (actionableelementid)" # change per KC run
correct_col = "eventualcorrect"
student_col = "Anon Student Id"

fixed_fold_file = "data/fixed_outer_folds_af.csv"
oof_output_dir = "DKT_AF"
os.makedirs(oof_output_dir, exist_ok=True)

# logging_data must already be loaded before running this script/cell.
# Example:
# logging_data = pd.read_csv("feedbook_firstattempt.csv")


# ============================================================
# 3. Preserve original row IDs BEFORE model-specific filtering
# ============================================================
logging_data = logging_data.copy().reset_index(drop=True)

if "row_id" not in logging_data.columns:
    logging_data["row_id"] = np.arange(len(logging_data), dtype=np.int64)

# Avoid merge suffixes if the same in-memory dataframe was used previously.
if "outer_fold" in logging_data.columns:
    logging_data = logging_data.drop(columns=["outer_fold"])


# ============================================================
# 4. Load/create fixed STUDENT-GROUPED outer folds
# ============================================================
if os.path.exists(fixed_fold_file):
    fixed_folds = pd.read_csv(fixed_fold_file)

    required = {"row_id", "outer_fold"}
    if not required.issubset(fixed_folds.columns):
        raise ValueError(
            f"{fixed_fold_file} must contain columns 'row_id' and 'outer_fold'."
        )

    if fixed_folds["row_id"].duplicated().any():
        raise ValueError(f"{fixed_fold_file} contains duplicate row_id values.")

    found_folds = sorted(
        fixed_folds["outer_fold"].dropna().astype(int).unique().tolist()
    )
    expected_folds = list(range(1, n_splits + 1))
    if found_folds != expected_folds:
        raise ValueError(
            f"{fixed_fold_file} contains folds {found_folds}, "
            f"but this run expects {expected_folds}."
        )

    logging_data = logging_data.merge(
        fixed_folds[["row_id", "outer_fold"]],
        on="row_id",
        how="left",
        validate="one_to_one",
    )

    if logging_data["outer_fold"].isna().any():
        raise ValueError(
            "Some row_id values are missing outer_fold assignments. "
            "The fixed-fold file may belong to a different dataset."
        )

    logging_data["outer_fold"] = logging_data["outer_fold"].astype(int)
    print(f"Loaded fixed folds from {fixed_fold_file}")

else:
    base_for_folds = logging_data.dropna(subset=[student_col]).copy()

    gkf = GroupKFold(n_splits=n_splits)
    base_for_folds["outer_fold"] = -1

    for fold_idx, (_, test_idx) in enumerate(
        gkf.split(base_for_folds, groups=base_for_folds[student_col]),
        start=1,
    ):
        base_for_folds.iloc[
            test_idx,
            base_for_folds.columns.get_loc("outer_fold"),
        ] = fold_idx

    fixed_folds = base_for_folds[["row_id", "outer_fold"]].copy()
    fixed_folds.to_csv(fixed_fold_file, index=False)

    logging_data = logging_data.merge(
        fixed_folds,
        on="row_id",
        how="left",
        validate="one_to_one",
    )

    print(f"Saved fixed folds to {fixed_fold_file}")


# ============================================================
# 5. Model-specific preprocessing
# ============================================================
logging_model = logging_data.dropna(
    subset=[question_col, correct_col, student_col, "outer_fold"]
).copy()

logging_model[correct_col] = logging_model[correct_col].astype(int)
logging_model["outer_fold"] = logging_model["outer_fold"].astype(int)

if not logging_model[correct_col].isin([0, 1]).all():
    raise ValueError(f"{correct_col} must contain only 0/1 values after filtering.")

# Validate that a student never appears in more than one fixed outer fold.
student_fold_counts = logging_model.groupby(student_col)["outer_fold"].nunique()
if (student_fold_counts > 1).any():
    bad_students = student_fold_counts[student_fold_counts > 1].index.tolist()[:10]
    raise ValueError(
        "Fixed outer folds are not student-grouped. Example offending students: "
        f"{bad_students}"
    )

# row_id -> student ID for OOF files
student_id_lookup = (
    logging_model[["row_id", student_col]]
    .drop_duplicates(subset=["row_id"])
    .set_index("row_id")[student_col]
    .to_dict()
)

# Global KC/question mapping. Reserve 0 for padding.
all_qids = logging_model[question_col].dropna().unique()
qid_to_index = {qid: idx + 1 for idx, qid in enumerate(all_qids)}
num_questions = len(qid_to_index) + 1

print(f"Model: {model_name}")
print(f"KC: {kc_name}")
print(f"Target: {correct_col}")
print(f"seq_len: {seq_len}")
print(f"num_questions (including padding 0): {num_questions}")


# ============================================================
# 6. Dataset
# ============================================================
class KTDataFromLogging(Dataset):
    def __init__(
        self,
        df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index,
    ):
        self.seq_len = seq_len
        self.samples = []

        df = df.copy()
        df["qid_index"] = (
            df[question_col].map(qid_to_index).fillna(0).astype(int)
        )

        # Preserve existing row order within each student.
        # Make sure logging_data is already in chronological interaction order.
        for _, group in df.groupby(student_col, sort=False):
            q_seq = group["qid_index"].tolist()
            r_seq = group[correct_col].astype(int).tolist()
            row_seq = group["row_id"].astype(int).tolist()

            # Adjacent windows share one interaction so every target retains context.
            for start in range(0, len(q_seq), seq_len - 1):
                end = min(start + seq_len, len(q_seq))

                if end - start < 2:
                    break

                q_chunk = q_seq[start:end]
                r_chunk = r_seq[start:end]
                row_chunk = row_seq[start:end]

                pad_len = seq_len - len(q_chunk)
                if pad_len > 0:
                    q_chunk += [0] * pad_len
                    r_chunk += [0] * pad_len
                    row_chunk += [-1] * pad_len

                self.samples.append(
                    (
                        torch.tensor(q_chunk, dtype=torch.long),
                        torch.tensor(r_chunk, dtype=torch.long),
                        torch.tensor(row_chunk, dtype=torch.long),
                    )
                )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


# ============================================================
# 7. DKT helpers: pyKT-equivalent next-interaction alignment
# ============================================================
def dkt_selected_predictions(model, q_batch, r_batch):
    """
    outputs[:, t, k] = probability after interaction t for KC k.
    For target interaction t+1, select KC q[t+1] from output t.
    """
    outputs = model(q_batch, r_batch)  # [B, L, num_c]
    next_q = q_batch[:, 1:]            # [B, L-1]
    selected = outputs[:, :-1, :].gather(
        dim=2,
        index=next_q.unsqueeze(-1),
    ).squeeze(-1)                       # [B, L-1]
    return selected, next_q


def collect_predictions(model, data_loader, include_rows=False):
    model.eval()
    preds, labels, row_ids = [], [], []

    with torch.no_grad():
        for q_batch, r_batch, rowid_batch in data_loader:
            q_batch = q_batch.to(device)
            r_batch = r_batch.to(device)

            selected, next_q = dkt_selected_predictions(
                model, q_batch, r_batch
            )
            next_r = r_batch[:, 1:]
            valid_mask = next_q != 0

            if valid_mask.any():
                preds.extend(selected[valid_mask].detach().cpu().numpy().tolist())
                labels.extend(next_r[valid_mask].detach().cpu().numpy().tolist())

                if include_rows:
                    row_mask_cpu = valid_mask.detach().cpu()
                    next_rows = rowid_batch[:, 1:]
                    these_rows = next_rows[row_mask_cpu].numpy().tolist()
                    if any(rid == -1 for rid in these_rows):
                        raise RuntimeError("A padded row_id passed the valid DKT mask.")
                    row_ids.extend(these_rows)

    if include_rows:
        return preds, labels, row_ids
    return preds, labels


def safe_auc(labels, preds):
    if len(labels) == 0 or len(np.unique(labels)) < 2:
        return np.nan
    return roc_auc_score(labels, preds)


def compute_metrics(labels, preds):
    labels = np.asarray(labels, dtype=int)
    preds = np.asarray(preds, dtype=float)
    binary = (preds > 0.5).astype(int)

    return {
        "auc": safe_auc(labels, preds),
        "accuracy": accuracy_score(labels, binary),
        "rmse": root_mean_squared_error(labels, preds),
        "mae": mean_absolute_error(labels, preds),
        "precision": precision_score(labels, binary, zero_division=0),
        "recall": recall_score(labels, binary, zero_division=0),
        "f1": f1_score(labels, binary, zero_division=0),
    }


# ============================================================
# 8. Hyperparameter grid (unchanged from supplied DKT code)
# ============================================================
param_search_space = {
    "emb_size": [64, 128, 256],
    "learning_rate": [1e-2, 1e-3, 1e-4],
    "dropout": [0.1, 0.3, 0.5],
}


# ============================================================
# 9. OOF and metric holders
# ============================================================
all_preds_all_folds = []
all_labels_all_folds = []
all_rowids_all_folds = []
all_foldnums_all_folds = []

auc_per_fold = []
acc_per_fold = []
rmse_per_fold = []
mae_per_fold = []
precision_per_fold = []
recall_per_fold = []
f1_per_fold = []


# ============================================================
# 10. Outer CV using fixed folds
# ============================================================
for fold in range(1, n_splits + 1):
    print(f"\n=== Outer Fold {fold}/{n_splits} ===")

    train_val_df = logging_model[
        logging_model["outer_fold"] != fold
    ].copy()

    test_df = logging_model[
        logging_model["outer_fold"] == fold
    ].copy()

    # -------------------------
    # Inner CV hyperparameter tuning
    # -------------------------
    inner_cv = GroupKFold(n_splits=n_splits)
    inner_groups = train_val_df[student_col]

    def objective(trial):
        emb_size = trial.suggest_categorical(
            "emb_size", param_search_space["emb_size"]
        )
        lr = trial.suggest_categorical(
            "learning_rate", param_search_space["learning_rate"]
        )
        dropout = trial.suggest_categorical(
            "dropout", param_search_space["dropout"]
        )

        auc_scores = []
        best_epochs = []

        for inner_train_idx, inner_val_idx in inner_cv.split(
            train_val_df,
            groups=inner_groups,
        ):
            inner_train_df = train_val_df.iloc[inner_train_idx]
            inner_val_df = train_val_df.iloc[inner_val_idx]

            train_dataset = KTDataFromLogging(
                inner_train_df,
                seq_len,
                question_col,
                correct_col,
                student_col,
                qid_to_index,
            )
            val_dataset = KTDataFromLogging(
                inner_val_df,
                seq_len,
                question_col,
                correct_col,
                student_col,
                qid_to_index,
            )

            train_loader = DataLoader(
                train_dataset,
                batch_size=batch_size,
                shuffle=True,
            )
            val_loader = DataLoader(
                val_dataset,
                batch_size=batch_size,
                shuffle=False,
            )

            model = DKT(
                num_c=num_questions,
                emb_size=emb_size,
                dropout=dropout,
                emb_type="qid",
            ).to(device)

            criterion = nn.BCELoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)

            best_auc_inner = -np.inf
            best_epoch_inner = 1
            no_improve = 0

            for epoch in range(max_epochs):
                model.train()

                for q_batch, r_batch, rowid_batch in train_loader:
                    q_batch = q_batch.to(device)
                    r_batch = r_batch.to(device)

                    optimizer.zero_grad()

                    selected_preds, next_q = dkt_selected_predictions(
                        model, q_batch, r_batch
                    )
                    next_r = r_batch[:, 1:].float()
                    valid_mask = next_q != 0

                    # A chunk of length 1 has no next-interaction target.
                    # Skip an all-singleton batch rather than taking BCE on empty tensors.
                    if not valid_mask.any():
                        continue

                    loss = criterion(
                        selected_preds[valid_mask],
                        next_r[valid_mask],
                    )

                    loss.backward()
                    optimizer.step()

                val_preds, val_labels = collect_predictions(
                    model, val_loader, include_rows=False
                )
                val_auc = safe_auc(val_labels, val_preds)

                if not np.isnan(val_auc):
                    if val_auc > best_auc_inner:
                        best_auc_inner = val_auc
                        best_epoch_inner = epoch + 1
                        no_improve = 0
                    else:
                        no_improve += 1

                    if no_improve >= patience:
                        break

            auc_scores.append(best_auc_inner)
            best_epochs.append(best_epoch_inner)

        trial.set_user_attr(
            "recommended_epochs",
            max(1, int(np.median(best_epochs))),
        )

        return np.mean(auc_scores) if auc_scores else 0.0

    sampler = GridSampler(param_search_space)
    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
    )

    grid_size = 1
    for values in param_search_space.values():
        grid_size *= len(values)

    study.optimize(
        objective,
        n_trials=grid_size,
        show_progress_bar=True,
    )

    best_params = study.best_params
    recommended_epochs = int(
        study.best_trial.user_attrs["recommended_epochs"]
    )
    print(
        f"\nBest params for fold {fold}: {best_params}, "
        f"AUC: {study.best_value:.6f}"
    )

    # -------------------------
    # Retrain best model on outer train+validation students
    # -------------------------
    train_dataset = KTDataFromLogging(
        train_val_df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index,
    )
    test_dataset = KTDataFromLogging(
        test_df,
        seq_len,
        question_col,
        correct_col,
        student_col,
        qid_to_index,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    best_model = DKT(
        num_c=num_questions,
        emb_size=best_params["emb_size"],
        dropout=best_params["dropout"],
        emb_type="qid",
    ).to(device)

    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(
        best_model.parameters(),
        lr=best_params["learning_rate"],
    )

    # Retrain on every outer-training student for the epoch count
    # selected from the inner validation folds.
    for epoch in range(recommended_epochs):
        best_model.train()

        for q_batch, r_batch, rowid_batch in train_loader:
            q_batch = q_batch.to(device)
            r_batch = r_batch.to(device)

            optimizer.zero_grad()

            selected_preds, next_q = dkt_selected_predictions(
                best_model, q_batch, r_batch
            )
            next_r = r_batch[:, 1:].float()
            valid_mask = next_q != 0

            if not valid_mask.any():
                continue

            loss = criterion(
                selected_preds[valid_mask],
                next_r[valid_mask],
            )

            loss.backward()
            optimizer.step()

        # The epoch count is selected only from inner validation folds.
        # The outer test fold is not inspected during training.

    # -------------------------
    # Final outer-test OOF predictions
    # -------------------------
    fold_preds, fold_labels, fold_row_ids = collect_predictions(
        best_model,
        test_loader,
        include_rows=True,
    )

    if len(fold_labels) == 0:
        print(f"WARNING: Fold {fold} produced no scorable predictions.")
        continue

    if len(fold_row_ids) != len(set(fold_row_ids)):
        raise ValueError(f"Duplicate row_id values inside OOF fold {fold}.")

    metrics = compute_metrics(fold_labels, fold_preds)

    auc_per_fold.append(metrics["auc"])
    acc_per_fold.append(metrics["accuracy"])
    rmse_per_fold.append(metrics["rmse"])
    mae_per_fold.append(metrics["mae"])
    precision_per_fold.append(metrics["precision"])
    recall_per_fold.append(metrics["recall"])
    f1_per_fold.append(metrics["f1"])

    print(f"\nEvaluation on Fold {fold} Test Set:")
    print(f"AUC: {metrics['auc']:.6f}")
    print(f"Accuracy: {metrics['accuracy']:.6f}")
    print(f"RMSE: {metrics['rmse']:.6f}")
    print(f"MAE: {metrics['mae']:.6f}")
    print(f"Precision: {metrics['precision']:.6f}")
    print(f"Recall: {metrics['recall']:.6f}")
    print(f"F1 Score: {metrics['f1']:.6f}")

    fold_df = pd.DataFrame({
        "row_id": fold_row_ids,
        "student_id": [student_id_lookup[row_id] for row_id in fold_row_ids],
        "fold": fold,
        "y_true": fold_labels,
        "y_pred": fold_preds,
        "model_name": model_name,
        "kc_name": kc_name,
    }).sort_values("row_id").reset_index(drop=True)

    fold_outfile = os.path.join(
        oof_output_dir,
        f"oof_{model_name}_{kc_name}_fold{fold}.csv",
    )
    fold_df.to_csv(fold_outfile, index=False)
    print(f"Saved Fold {fold} OOF predictions to {fold_outfile}")

    all_preds_all_folds.extend(fold_preds)
    all_labels_all_folds.extend(fold_labels)
    all_rowids_all_folds.extend(fold_row_ids)
    all_foldnums_all_folds.extend([fold] * len(fold_row_ids))


# ============================================================
# 11. Combined OOF
# ============================================================
oof_df = pd.DataFrame({
    "row_id": all_rowids_all_folds,
    "student_id": [student_id_lookup[row_id] for row_id in all_rowids_all_folds],
    "fold": all_foldnums_all_folds,
    "y_true": all_labels_all_folds,
    "y_pred": all_preds_all_folds,
    "model_name": model_name,
    "kc_name": kc_name,
}).sort_values("row_id").reset_index(drop=True)

if oof_df["row_id"].duplicated().any():
    dupes = oof_df.loc[oof_df["row_id"].duplicated(), "row_id"].head(10).tolist()
    raise ValueError(f"Duplicate row_id values in combined OOF: {dupes}")

# Check that saved fold number agrees with the fixed outer-fold assignment.
expected_fold_map = logging_model.set_index("row_id")["outer_fold"].to_dict()
wrong_fold = [
    row_id
    for row_id, fold in zip(oof_df["row_id"], oof_df["fold"])
    if expected_fold_map[row_id] != fold
]
if wrong_fold:
    raise ValueError(
        "OOF fold labels disagree with fixed outer folds. "
        f"Example row_ids: {wrong_fold[:10]}"
    )

oof_outfile = os.path.join(
    oof_output_dir,
    f"oof_{model_name}_{kc_name}_all.csv",
)
oof_df.to_csv(oof_outfile, index=False)
print(f"\nSaved combined OOF predictions to {oof_outfile}")


# ============================================================
# 12. Pooled + fold-average metrics
# ============================================================
if len(oof_df) > 0:
    pooled = compute_metrics(oof_df["y_true"].values, oof_df["y_pred"].values)

    print("\n=== Pooled OOF Results ===")
    print(f"Pooled AUC: {pooled['auc']:.6f}")
    print(f"Pooled Accuracy: {pooled['accuracy']:.6f}")
    print(f"Pooled RMSE: {pooled['rmse']:.6f}")
    print(f"Pooled MAE: {pooled['mae']:.6f}")
    print(f"Pooled Precision: {pooled['precision']:.6f}")
    print(f"Pooled Recall: {pooled['recall']:.6f}")
    print(f"Pooled F1: {pooled['f1']:.6f}")


def print_mean_sd(name, values):
    vals = [v for v in values if not np.isnan(v)]
    if not vals:
        print(f"{name}: NA")
    elif len(vals) == 1:
        print(f"{name}: {vals[0]:.6f}")
    else:
        print(f"{name}: {mean(vals):.6f} ± {stdev(vals):.6f}")


print(f"\n=== Fold-Averaged Results ({n_splits}-Fold CV) ===")
print_mean_sd("AUC", auc_per_fold)
print_mean_sd("Accuracy", acc_per_fold)
print_mean_sd("RMSE", rmse_per_fold)
print_mean_sd("MAE", mae_per_fold)
print_mean_sd("Precision", precision_per_fold)
print_mean_sd("Recall", recall_per_fold)
print_mean_sd("F1", f1_per_fold)